In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Colab Notebooks/AI/'

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F 

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # --- 특징 추출기 (Feature Extractor) ---
        self.features = nn.Sequential(
            # 1. 첫 번째 합성곱 블록
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # 2. 두 번째 합성곱 블록
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # --- 분류기 (Classifier) ---
        self.classifier = nn.Sequential(
            nn.Linear(in_features=32 * 8 * 8, out_features=256),
            nn.ReLU(),
            nn.Linear(in_features=256, out_features=256),
            nn.ReLU(),
            nn.Linear(in_features=256, out_features=num_classes)
        )

    def forward(self, x):
        # 1. 특징 추출
        x = self.features(x) 
        # 2. 평탄화 
        x = torch.flatten(x, start_dim=1)
        # 3. 분류기 통과
        x = self.classifier(x)
        return x

# 3. 성능 향상을 위한 추가 기법 적용

- 모델의 성능은 아키텍쳐뿐만 아니라, **데이터의 질**과 학습 **과정의 안정성**에 좌우 됨
- 데이터를 더 수집하고 전처리 하기는 어려우니, 여기서는 **데이터 증강**을 활용
- 학습을 안정적으로 최적화 하기 위해 **학습률 스케쥴러**를 도입

## 3-1. 데이터 증강(Data Augmentation)

- 데이터 증강은 가지고 있는 훈련 데이터에 **무작위 변형**을 가하는 방법
- 가지고 있는 데이터만 가지고, 더 많은 학습을 한 것과 같은 효과를 기대할 수 있음
    - 동일한 이미지를 조금씩 다른 형태로 학습하게 하면, 더욱 본질적인 패턴에 집중 할 수 있음

### 3-1-1. 코드 구현

- `transforms.Compose`으로 데이터 변환 방식을 조금 더 상세하게 정의
- **RandomCrop:** 이미지 주변에 정해진 픽셀만큼 패딩 후 무작위로 위치를 자르기
- **RadnomHorizontalFlip:** 확률로 좌우 반전

In [ ]:
import torchvision
import torchvision.transforms as transforms

# 테스트용 데이터에는 기존의 transform을 그대로 사용
transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616))
])

# 테스트용(test) 데이터셋 로드
testset = torchvision.datasets.CIFAR10(root=base_path + './data', train=False,
                                       download=True, transform=transform)

# 데이터 증강이 포함된 새로운 변환(transform) 정의
train_transform_aug = transforms.Compose([
    # 32x32 이미지 주변에 4픽셀 패딩 후 무작위로 32x32 자르기
    transforms.RandomCrop(32, padding=4),       
    # 50% 확률로 좌우 반전
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616))
])

# 훈련용(train) 데이터셋 로드
trainset = torchvision.datasets.CIFAR10(root=base_path + './data', train=True,
                                        download=True, transform=train_transform_aug)

# 훈련용 DataLoader 생성
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                          shuffle=True)
# 테스트용 DataLoader 생성
testloader = torch.utils.data.DataLoader(testset, batch_size=128,
                                         shuffle=False)


## 3-2. 학습률 스케쥴러

- 학습률이 챕터2에서는 0.001로 고정이었음.
- 오차의 크기와 상관없이 항상 고정된 학습률로 진행하여 학습의 효율성이 떨어짐.
- 차라리, 학습 초반에는 큰 보폭으로 빠르게 최적 지점을 찾고, 점진적으로 학습률을 줄이는 방식으로 세밀하게 탐색하는 것이 효과정

### 3-2-1. 유용한 학습률 스케줄러

1. **StepLR:** 일정 주기마다 학습률을 정해진 비율로 감소
2. **ExponentialLR:** 매 에포크마다 학습률을 지수적으로 감소
3. **CosineAnnealingLR:** 코사인 함수의 반주기 형태로 감소시키고, 증가시키는 방식
4. **ReduceLROnPlateau:** 모델의 성능 지표가 일정 기간 동안 개선되지 않을 때, 학습률을 감소

In [ ]:
import torch.optim as optim

# 장치 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("사용 장치:", device)

# 모델과 옵티마이저를 다시 정의 (이전 상태에서 이어서 할 수도 있음)
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# StepLR 스케줄러 정의
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 3-3. 다시 학습

- 에포크가 끝날때 마다 scheduler 호출

In [ ]:
from tqdm import tqdm

num_epochs = 15 # 스케줄러 효과를 보기 위해 에포크 수를 늘림

for epoch in range(num_epochs):
    # --- 훈련(Train) 단계 ---
    model.train()
    running_loss = 0.0
    # 데이터 증강이 적용된 trainloader_aug 사용
    for i, data in enumerate(tqdm(trainloader), 0):
        inputs, labels = data[0].to(device), data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # --- 학습률 업데이트 ---
    # 매 에포크가 끝난 후 스케줄러를 업데이트
    scheduler.step() 

    # 현재 학습률 출력
    current_lr = optimizer.param_groups[0]['lr']
    print(f'[{epoch + 1}] 훈련 손실: {running_loss / len(trainloader):.3f}, 현재 학습률: {current_lr}')

    # --- 평가(Evaluation) 단계 ---
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data[0].to(device), data[1].to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'[{epoch + 1}] 테스트 정확도: {100 * correct / total:.2f} %')

print('학습 종료')

## 3-4. 다시... 평가

In [ ]:
from PIL import Image

# convert('RGB')는 이미지가 4채널(RGBA)일 경우를 대비해 3채널(RGB)로 통일해주는 역할
image = Image.open(base_path + 'cat.jpg').convert('RGB')

# 이미지 확인
# image.show() 

# 훈련 시 사용했던 전처리 과정을 그대로 정의
inference_transform = transforms.Compose([
    transforms.Resize((32, 32)), # 1. 32x32 크기로 리사이즈
    transforms.ToTensor(),       # 2. 텐서로 변환
    # 3. 훈련 때와 "동일한" 값으로 정규화
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2470, 0.2435, 0.2616))
])

# 전처리 적용
input_tensor = inference_transform(image)

# 전처리 된 이미지 확인
print(input_tensor.shape) # torch.Size([3, 32, 32])

In [ ]:
import torch

# 1. 모델을 평가 모드로 전환
model.eval()

# 2. 배치 차원 추가 및 장치로 이동
input_batch = input_tensor.unsqueeze(0).to(device)
# input_batch = input_tensor.unsqueeze(0)

# 3. 기울기 계산을 하지 않도록 설정
with torch.no_grad():
    output = model(input_batch)

# 4. 출력(Logits)을 확률(Probabilities)로 변환
# Softmax 함수는 모든 클래스에 대한 점수의 합이 1이 되도록 만듬
probabilities = torch.nn.functional.softmax(output[0], dim=0)

# 5. 가장 확률이 높은 클래스 찾기
top_prob, top_catid = torch.max(probabilities, 0)
predicted_idx = top_catid.item()

# CIFAR-10 클래스 이름 가져오기
class_names = trainset.classes # ['airplane', 'automobile', 'bird', 'cat', ...]
predicted_label = class_names[predicted_idx]

print(f"모델의 예측: '{predicted_label}' (신뢰도: {top_prob.item()*100:.2f}%)")

# 시각화로 최종 확인
import matplotlib.pyplot as plt

plt.imshow(image)
plt.title(f"Prediction: {predicted_label} ({top_prob.item()*100:.2f}%)")
plt.axis('off')
plt.show()